In [1]:
# %% 셀 1: eval.jsonl 파싱 + 채널 통계 (나중에 bucket별 분석용)
import os
import re
import json
import glob
import random
import hashlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
INST_DIR   = "./data/6_telop_instances"

N_BUCKETS  = 5  # 나중에 평가 시 bucket별 분석용
SEED       = 42


# -------------------------
# 1) eval.jsonl에서 전체 채널 목록 추출
# -------------------------
def extract_user_text(msg_content) -> str:
    if isinstance(msg_content, str):
        return msg_content
    if isinstance(msg_content, list):
        return "\n".join(
            part.get("text", "") for part in msg_content if isinstance(part, dict)
        )
    return ""


eval_channels: set[str] = set()
with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in tqdm(lines, desc="eval.jsonl 파싱"):
    ex = json.loads(line)
    user_msg = next(m for m in ex["messages"] if m["role"] == "user")
    text = extract_user_text(user_msg["content"])
    m_ch = re.search(r"채널:\s*([^\n]+)", text)
    if m_ch:
        eval_channels.add(m_ch.group(1).strip())

all_channels = sorted(eval_channels)
print(f"\neval.jsonl의 고유 채널 수: {len(all_channels)}")


# -------------------------
# 2) 채널별 통계 (나중에 bucket별 분석용)
# -------------------------
def channel_stats(channel: str) -> dict | None:
    ch_dir = os.path.join(INST_DIR, channel)
    if not os.path.isdir(ch_dir):
        return None

    json_paths = sorted(glob.glob(os.path.join(glob.escape(ch_dir), "*.json")))
    if not json_paths:
        return None

    n_videos    = 0
    n_instances = 0
    dur_sum     = 0.0
    textlen_sum = 0
    density_sum = 0.0

    for jp in json_paths:
        try:
            with open(jp, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception:
            continue

        instances = data.get("instances", [])
        n_videos += 1

        if not instances:
            continue

        starts = np.array([inst["start_sec"] for inst in instances], dtype=np.float64)
        ends   = np.array([inst["end_sec"]   for inst in instances], dtype=np.float64)
        texts  = [inst["text"] for inst in instances]

        durs    = np.clip(ends - starts, 0, None)
        vid_len = float(ends.max()) if len(ends) else 0.0

        n_instances += len(instances)
        dur_sum     += float(durs.sum())
        textlen_sum += sum(len(t) for t in texts)

        if vid_len > 0:
            density_sum += float(durs.sum()) / vid_len

    if n_videos == 0:
        return None

    return {
        "n_videos":     n_videos,
        "n_instances":  n_instances,
        "avg_count":    n_instances / n_videos,
        "avg_duration": (dur_sum / n_instances) if n_instances > 0 else 0.0,
        "avg_text_len": (textlen_sum / n_instances) if n_instances > 0 else 0.0,
        "avg_density":  density_sum / n_videos,
    }


stats_rows = []
for ch in tqdm(all_channels, desc="채널 통계 계산"):
    s = channel_stats(ch)
    if s is None:
        continue
    stats_rows.append({"channel": ch, **s})

stats_df = pd.DataFrame(stats_rows)
stats_df = stats_df.sort_values("avg_density").reset_index(drop=True)
stats_df["bucket"] = pd.qcut(
    stats_df["avg_density"],
    q=N_BUCKETS,
    labels=list(range(N_BUCKETS)),
).astype(int)

print(f"\n✅ 전체 채널: {len(stats_df)}개")
print(stats_df.groupby("bucket")["avg_density"].agg(["count", "min", "max", "mean"]))


# -------------------------
# 3) 영상별 swap 채널 미리 결정 (재현성 보장)
# -------------------------
def get_swap_channel(orig_channel: str, video_name: str, candidates: list[str]) -> str:
    """hash 기반으로 결정론적 swap 채널 선택. 자기 자신 제외."""
    h = hashlib.sha256(f"{orig_channel}||{video_name}".encode()).hexdigest()
    seed = int(h[:8], 16)
    rng = random.Random(seed)
    pool = [c for c in candidates if c != orig_channel]
    return rng.choice(pool)


# 테스트
_test_ch = all_channels[0]
_test_vn = "test_video"
_test_swap = get_swap_channel(_test_ch, _test_vn, all_channels)
print(f"\nswap 테스트: {_test_ch!r} → {_test_swap!r}")

print(f"\n✅ 다음 셀에서 사용할 변수: all_channels, stats_df, get_swap_channel")

/home/taeyoung/miniconda3/envs/annotation/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
eval.jsonl 파싱: 100%|██████████| 3320/3320 [00:00<00:00, 68164.68it/s]



eval.jsonl의 고유 채널 수: 664


채널 통계 계산: 100%|██████████| 664/664 [00:10<00:00, 63.98it/s]


✅ 전체 채널: 664개
        count       min        max      mean
bucket                                      
0         133  0.011535   0.664213  0.294376
1         133  0.685286   1.688121  1.160495
2         132  1.689540   3.240800  2.520000
3         133  3.257812   4.646314  3.939302
4         133  4.648890  16.820411  6.215873

swap 테스트: '\x08고기남자' → 'MBN MUSIC'

✅ 다음 셀에서 사용할 변수: all_channels, stats_df, get_swap_channel


In [2]:
# %% 셀 2: 기존 sglang 서버 종료 + ep3 merged 모델 서버 시작
import os
import subprocess
import time
import requests
import signal

os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

MODEL_PATH      = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep3-merged"
PORT            = 8000
SERVER_LOG      = "sglang_server.log"
CONTEXT_LENGTH  = 131072


# -------------------------
# 1) 기존 서버 프로세스 종료
# -------------------------
print("🛑 기존 sglang 서버 종료 시도...")

if "server_process" in globals() and server_process.poll() is None:
    try:
        server_process.terminate()
        server_process.wait(timeout=10)
        print("   ✅ 이전 server_process 핸들로 종료")
    except Exception as e:
        try:
            server_process.kill()
            print(f"   ⚠️ terminate 실패 → kill ({e})")
        except Exception:
            pass

try:
    out = subprocess.run(
        ["lsof", "-t", f"-i:{PORT}"],
        capture_output=True, text=True, timeout=5,
    ).stdout.strip()
    pids = [int(p) for p in out.splitlines() if p.strip().isdigit()]
    for pid in pids:
        try:
            os.kill(pid, signal.SIGTERM)
            print(f"   ✅ PID {pid} SIGTERM")
        except ProcessLookupError:
            pass
    time.sleep(3)
    out2 = subprocess.run(
        ["lsof", "-t", f"-i:{PORT}"],
        capture_output=True, text=True, timeout=5,
    ).stdout.strip()
    for p in out2.splitlines():
        if p.strip().isdigit():
            try:
                os.kill(int(p), signal.SIGKILL)
                print(f"   ✅ PID {p} SIGKILL")
            except ProcessLookupError:
                pass
except FileNotFoundError:
    print("   ⚠️ lsof 없음 — 포트 점유 확인 skip")

for _ in range(10):
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=1)
        if r.status_code == 200:
            time.sleep(1)
            continue
    except Exception:
        break
print("   ✅ 포트 정리 완료\n")


# -------------------------
# 2) 서버 시작
# -------------------------
assert os.path.isdir(MODEL_PATH), f"모델 경로 없음: {MODEL_PATH}"

sglang_log = open(SERVER_LOG, "w")

server_process = subprocess.Popen(
    [
        "python", "-m", "sglang.launch_server",
        "--model-path", MODEL_PATH,
        "--port", str(PORT),
        "--mem-fraction-static", "0.8",
        "--context-length", str(CONTEXT_LENGTH),
        "--reasoning-parser", "qwen3",
        "--attention-backend", "triton",
    ],
    stdout=sglang_log,
    stderr=subprocess.STDOUT,
)

print(f"⏳ SGLang 서버 시작 중...")
print(f"   모델: {MODEL_PATH}")
print(f"   context-length: {CONTEXT_LENGTH}")

ready = False
for i in range(600):
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=1)
        if r.status_code == 200:
            print(f"✅ 서버 준비 완료! ({i}초)")
            ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    print(f"❌ 서버 시작 실패. 로그 확인: tail -n 200 {SERVER_LOG}")

🛑 기존 sglang 서버 종료 시도...
   ✅ PID 2766060 SIGTERM
   ✅ PID 2766060 SIGKILL
   ✅ 포트 정리 완료

⏳ SGLang 서버 시작 중...
   모델: ./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep3-merged
   context-length: 131072
✅ 서버 준비 완료! (24초)


In [ ]:
# %% 셀 3: 664채널 × 5영상 × (본채널 + 랜덤1채널) inference
import os
import re
import json
import time
import httpx
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

EVAL_JSONL  = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR  = "./data/7_qwen_test"
MODEL_NAME  = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep3-merged"

TEMPERATURE = 0.1
TOP_P       = 1.0
MAX_TOKENS  = 126976

MAX_WORKERS = 1
TIMEOUT     = 3600

os.makedirs(OUTPUT_DIR, exist_ok=True)

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    timeout=httpx.Timeout(TIMEOUT, connect=5.0),
)


# -------------------------
# 1) eval.jsonl 전체 파싱
# -------------------------
def parse_user_text(msg_content) -> str:
    if isinstance(msg_content, str):
        return msg_content
    if isinstance(msg_content, list):
        return "\n".join(
            part.get("text", "") for part in msg_content if isinstance(part, dict)
        )
    return ""


with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    eval_lines = f.readlines()

target_entries = []
for line in tqdm(eval_lines, desc="eval.jsonl 파싱"):
    ex = json.loads(line)
    msgs = ex["messages"]
    sys_msg  = next(m for m in msgs if m["role"] == "system")
    user_msg = next(m for m in msgs if m["role"] == "user")
    user_text = parse_user_text(user_msg["content"])

    m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
    m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
    if not m_ch or not m_vn:
        continue

    channel = m_ch.group(1).strip()
    video   = m_vn.group(1).strip()

    target_entries.append({
        "channel":    channel,
        "video_name": video,
        "system":     sys_msg["content"],
        "user_text":  user_text,
    })

print(f"\n✅ 대상 영상 수: {len(target_entries)}")


# -------------------------
# 2) job 리스트 구성: 영상당 2건 (본채널 + 랜덤1채널)
# -------------------------
def swap_channel_line(user_text: str, new_channel: str) -> str:
    return re.sub(
        r"채널:\s*[^\n]+",
        f"채널: {new_channel}",
        user_text,
        count=1,
    )


def output_path(orig_ch: str, video: str, cond_ch: str) -> str:
    return os.path.join(OUTPUT_DIR, orig_ch, video, f"{cond_ch}.json")


jobs = []
for ent in tqdm(target_entries, desc="job 리스트 구성"):
    orig_ch = ent["channel"]
    video   = ent["video_name"]

    # (a) 본채널 조건
    out_a = output_path(orig_ch, video, orig_ch)
    if not os.path.exists(out_a):
        jobs.append({
            "orig_channel": orig_ch,
            "video_name":   video,
            "cond_channel": orig_ch,
            "system":       ent["system"],
            "user_text":    ent["user_text"],  # 원본 그대로
            "out_path":     out_a,
        })

    # (b) 랜덤 swap 채널 조건
    swap_ch = get_swap_channel(orig_ch, video, all_channels)
    out_b = output_path(orig_ch, video, swap_ch)
    if not os.path.exists(out_b):
        jobs.append({
            "orig_channel": orig_ch,
            "video_name":   video,
            "cond_channel": swap_ch,
            "system":       ent["system"],
            "user_text":    swap_channel_line(ent["user_text"], swap_ch),
            "out_path":     out_b,
        })

total_expected = len(target_entries) * 2
print(f"\n📋 실행할 job: {len(jobs)} (skip: {total_expected - len(jobs)})")
jobs.reverse()

# -------------------------
# 3) 단일 요청 처리
# -------------------------
def call_model(job: dict) -> dict:
    t0 = time.time()
    try:
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": job["system"]},
                {"role": "user",   "content": job["user_text"]},
            ],
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        elapsed = time.time() - t0

        raw = resp.choices[0].message.content.strip()
        cleaned = raw
        if cleaned.startswith("```"):
            cleaned = re.sub(r"^```[a-zA-Z]*\n?", "", cleaned)
            cleaned = re.sub(r"\n?```$", "", cleaned)

        parsed = json.loads(cleaned)

        os.makedirs(os.path.dirname(job["out_path"]), exist_ok=True)
        record = {
            "orig_channel":      job["orig_channel"],
            "video_name":        job["video_name"],
            "cond_channel":      job["cond_channel"],
            "model":             MODEL_NAME,
            "params": {
                "temperature":   TEMPERATURE,
                "top_p":         TOP_P,
                "max_tokens":    MAX_TOKENS,
            },
            "system":            job["system"],
            "user_text":         job["user_text"],
            "raw_output":        raw,
            "instances":         parsed.get("instances", []) if isinstance(parsed, dict) else parsed,
            "elapsed_sec":       round(elapsed, 2),
            "prompt_tokens":     getattr(resp.usage, "prompt_tokens", None),
            "completion_tokens": getattr(resp.usage, "completion_tokens", None),
        }
        with open(job["out_path"], "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)

        return {"success": True, "elapsed": elapsed}

    except json.JSONDecodeError as e:
        return {"success": False, "error": f"JSONDecodeError: {str(e)[:100]}",
                "elapsed": time.time() - t0}
    except Exception as e:
        return {"success": False, "error": str(e)[:200],
                "elapsed": time.time() - t0}


# -------------------------
# 4) 병렬 실행
# -------------------------
if not jobs:
    print("\n✅ 모든 job이 이미 완료 — 새로 생성할 것 없음")
else:
    n_success = 0
    n_fail    = 0
    errors    = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(call_model, j): j for j in jobs}

        with tqdm(total=len(jobs), desc="inference") as pbar:
            for fut in as_completed(futures):
                job = futures[fut]
                res = fut.result()
                if res["success"]:
                    n_success += 1
                else:
                    n_fail += 1
                    errors.append({
                        "orig":  job["orig_channel"],
                        "video": job["video_name"],
                        "cond":  job["cond_channel"],
                        "err":   res["error"],
                    })
                pbar.update(1)
                pbar.set_postfix(ok=n_success, fail=n_fail)

    print(f"\n📊 완료")
    print(f"  성공: {n_success}")
    print(f"  실패: {n_fail}")
    if errors:
        print(f"\n실패 샘플 (최대 10개):")
        for e in errors[:10]:
            print(f"  [{e['orig']}] {e['video']} ← {e['cond']}: {e['err']}")

eval.jsonl 파싱: 100%|██████████| 3320/3320 [00:00<00:00, 55719.75it/s]



✅ 대상 영상 수: 3320


job 리스트 구성: 100%|██████████| 3320/3320 [00:00<00:00, 3539.16it/s]



📋 실행할 job: 542 (skip: 6098)


inference:   0%|          | 1/542 [43:52<395:36:13, 2632.48s/it, fail=1, ok=0]